# SmartCrop AI: Pipeline 1 — CropHarvest Binary Crop Classification
## Earth Observation Time-Series Modeling (Crop vs. Non-Crop)

**Benchmark Reference:**  
*Tseng, G., Zvonkov, I., Duro, C. P., et al. (2021). CropHarvest: A global dataset for crop-type classification. NeurIPS 2021 Datasets and Benchmarks Track.*

---

### Pipeline Objectives & Standards
1. **Binary Classification Objective:**
   - Class 0: **Non-Crop**
   - Class 1: **Crop**
   - Target label: `is_crop` (100% verified ground truth across all 87,464 available arrays).
2. **Earth Observation Input Representation ($12 \times 18$):**
   - 12 monthly composite steps ($T=12$).
   - 18 calibrated channels ($C=18$): Sentinel-1 SAR (VV, VH), Sentinel-2 Multi-Spectral (10 bands), ERA5 Climate (4 variables), SRTM Topography (elevation, slope).
   - Normalization via official `normalizing_dict.h5`.
3. **Data Integrity & Leakage Prevention:**
   - Official global test split (`is_test: true`, 4,810 samples) is strictly isolated as held-out test data.
   - Internal training pool (`is_test: false`, 82,654 samples) is partitioned into stratified 80:20 Train/Validation splits (`seed=42`).
   - Dedicated regional evaluation on Togo (`togo-eval.h5`, 306 samples) and China (`china-crop.h5`, 4,397 samples).
4. **Manual Training Gate:**
   - `EXECUTE_TRAINING = False` by default to allow pre-flight hardware and data audits before starting execution.


In [ ]:
# Cell 1: Environment Setup, Dependencies & Hardware Inspection
import os
import sys
import json
import time
import math
import random
import shutil
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional
from collections import Counter

# Dependencies verification
try:
    import h5py
    import numpy as np
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    import sklearn
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    print("[Setup] Installing required dependencies...")
    !pip install -q h5py scikit-learn seaborn matplotlib
    import h5py
    import numpy as np
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns

# Deterministic Seeding
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Hardware Inspection
print("=" * 75)
print("  CROPHARVEST PIPELINE 1: BINARY CROP CLASSIFICATION ENVIRONMENT")
print("=" * 75)
print(f"Python Version      : {sys.version.split()[0]}")
print(f"PyTorch Version     : {torch.__version__}")
print(f"h5py Version        : {h5py.__version__}")
print(f"CUDA Available      : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    gpu_cap = torch.cuda.get_device_capability(0)
    print(f"GPU Device          : {gpu_name}")
    print(f"GPU VRAM            : {gpu_mem:.2f} GB")
    print(f"Compute Capability  : {gpu_cap[0]}.{gpu_cap[1]}")
    DEVICE = torch.device("cuda:0")
else:
    print("WARNING: CUDA is NOT available. Running on CPU.")
    DEVICE = torch.device("cpu")

IS_KAGGLE = Path("/kaggle").exists()
if IS_KAGGLE:
    OUTPUT_DIR = Path("/kaggle/working/models/cropharvest/binary")
else:
    OUTPUT_DIR = Path("models/cropharvest/binary")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Runtime Environment : {'Kaggle Kernel' if IS_KAGGLE else 'Local Workstation'}")
print(f"Output Directory    : {OUTPUT_DIR.resolve()}")
print("=" * 75)


In [ ]:
# Cell 2: Dynamic Dataset Path Discovery
# Searches /kaggle/input and local roots to locate arrays/, labels.geojson, and normalizing_dict.h5.

def discover_cropharvest_binary_paths() -> Dict[str, Any]:
    search_roots = []
    if Path("/kaggle/input").exists():
        search_roots.append(Path("/kaggle/input"))
        try:
            for item in Path("/kaggle/input").iterdir():
                if item.is_dir():
                    search_roots.append(item)
                    for sub in item.iterdir():
                        if sub.is_dir():
                            search_roots.append(sub)
        except Exception as e:
            print(f"[Warning] Error scanning /kaggle/input: {e}")

    search_roots.extend([
        Path("datasets/cropharvest"),
        Path("../datasets/cropharvest"),
        Path("../../datasets/cropharvest"),
        Path("."),
        Path("..")
    ])

    resolved = {}
    print("[Discovery] Searching for CropHarvest dataset components...")

    for root in search_roots:
        if not root.exists():
            continue
        arrays_cand = root / "arrays"
        geojson_cand = root / "labels.geojson"
        norm_cand = root / "normalizing_dict.h5"

        if arrays_cand.exists() and geojson_cand.exists():
            resolved["dataset_root"] = root.resolve()
            resolved["arrays_dir"] = arrays_cand.resolve()
            resolved["labels_geojson"] = geojson_cand.resolve()
            if norm_cand.exists():
                resolved["normalizing_dict"] = norm_cand.resolve()
            
            for testbed in ["togo-eval.h5", "china-crop.h5"]:
                tb_path = root / testbed
                if tb_path.exists():
                    resolved[testbed] = tb_path.resolve()
            break

    if "arrays_dir" not in resolved or "labels_geojson" not in resolved:
        raise FileNotFoundError("Could not locate CropHarvest arrays/ folder and labels.geojson.")

    return resolved

ch_paths = discover_cropharvest_binary_paths()
print("\n" + "=" * 75)
print("  CROPHARVEST PATH DISCOVERY SUCCESSFUL")
print("=" * 75)
print(f"  Dataset Root     : {ch_paths['dataset_root']}")
print(f"  Arrays Directory : {ch_paths['arrays_dir']}")
print(f"  Labels GeoJSON   : {ch_paths['labels_geojson']}")
print(f"  Normalizing Dict : {ch_paths.get('normalizing_dict')}")
print(f"  Togo Testbed     : {ch_paths.get('togo-eval.h5')}")
print(f"  China Testbed    : {ch_paths.get('china-crop.h5')}")
print("=" * 75)


In [ ]:
# Cell 3: Dataset Structure, File Integrity & Shape Audit
# Audits normalizing dictionary and arrays/ folder for 12x18 shape and 0 NaNs/Infs.

arrays_dir = ch_paths["arrays_dir"]
norm_path = ch_paths.get("normalizing_dict")

if norm_path and norm_path.exists():
    with h5py.File(norm_path, "r") as h:
        mean = h["mean"][:]
        std = h["std"][:]
    print(f"[Normalizer] Loaded 18-channel normalization parameters:")
    print(f"  Mean shape: {mean.shape} | Range: [{mean.min():.2f}, {mean.max():.2f}]")
    print(f"  Std shape : {std.shape}  | Range: [{std.min():.4f}, {std.max():.2f}]")
else:
    raise FileNotFoundError("normalizing_dict.h5 is required for CropHarvest preprocessing.")

all_array_files = list(arrays_dir.glob("*.h5"))
total_arrays = len(all_array_files)
print(f"\n[Arrays] Total available HDF5 array files: {total_arrays:,}")

# Sample verification
sample_size = min(1000, total_arrays)
random.seed(SEED)
sample_check = random.sample(all_array_files, sample_size)

shapes = set()
dtypes = set()
nan_count = 0
inf_count = 0
corrupt_count = 0

for p in sample_check:
    try:
        with h5py.File(p, "r") as h:
            arr = h["array"][:]
            shapes.add(arr.shape)
            dtypes.add(str(arr.dtype))
            if np.isnan(arr).any():
                nan_count += 1
            if np.isinf(arr).any():
                inf_count += 1
    except Exception:
        corrupt_count += 1

print(f"[Audit Sample: {sample_size:,} arrays]")
print(f"  Observed Array Shapes : {shapes} (Expected: (12, 18))")
print(f"  Observed Data Types   : {dtypes}")
print(f"  Files with NaNs       : {nan_count}")
print(f"  Files with Infs       : {inf_count}")
print(f"  Corrupt Files         : {corrupt_count}")
assert corrupt_count == 0, f"Detected {corrupt_count} corrupted files!"
assert shapes == {(12, 18)}, f"Unexpected shapes found: {shapes}"
print("  --> Array integrity check PASSED!")


In [ ]:
# Cell 4: Binary Label & Split Distribution Analysis
# Aligns labels.geojson with arrays/ files, verifies exactly 1-to-1 match.

geojson_path = ch_paths["labels_geojson"]
with open(geojson_path, "r", encoding="utf-8") as f:
    geo_data = json.load(f)

features = geo_data["features"]
print(f"Total features in labels.geojson: {len(features):,}")

available_array_stems = set(p.name for p in all_array_files)
matched_records = []
is_crop_counts = Counter()
is_test_counts = Counter()

for feat in features:
    props = feat["properties"]
    fn = f"{props['index']}_{props['dataset']}.h5"
    if fn in available_array_stems:
        matched_records.append({
            "filename": fn,
            "filepath": str((arrays_dir / fn).resolve()),
            "index": props["index"],
            "dataset": props["dataset"],
            "lat": float(props["lat"]),
            "lon": float(props["lon"]),
            "is_crop": int(props["is_crop"]),
            "is_test_official": bool(props.get("is_test", False))
        })
        is_crop_counts[props["is_crop"]] += 1
        is_test_counts[props["is_test"]] += 1

print(f"Matched records with local arrays: {len(matched_records):,} / {len(all_array_files):,} (100.0%)")

print("\n" + "=" * 70)
print("  BINARY CROP CLASSIFICATION: CLASS & SPLIT BREAKDOWN")
print("=" * 70)
print(f"  Class 0 (Non-Crop)            : {is_crop_counts[0]:,} ({is_crop_counts[0]/len(matched_records)*100:.1f}%)")
print(f"  Class 1 (Crop)                : {is_crop_counts[1]:,} ({is_crop_counts[1]/len(matched_records)*100:.1f}%)")
print(f"  Official Train/Val (is_test=F): {is_test_counts[False]:,} ({is_test_counts[False]/len(matched_records)*100:.1f}%)")
print(f"  Official Test (is_test=T)     : {is_test_counts[True]:,} ({is_test_counts[True]/len(matched_records)*100:.1f}%)")
print("=" * 70)

# Save Class Mapping
CLASS_MAPPING = {0: "non_crop", 1: "crop"}
with open(OUTPUT_DIR / "classes.json", "w", encoding="utf-8") as f:
    json.dump(CLASS_MAPPING, f, indent=2)
print(f"Class mapping saved to: {OUTPUT_DIR / 'classes.json'}")


In [ ]:
# Cell 5: Stratified Train/Val Partition & Official Test Isolation
# Requirement: Never use held-out test data for training or tuning.
# The official train/val pool (is_test=False, 82,654 samples) is partitioned
# into 80% Train and 20% Validation using stratified sampling with SEED=42.

VAL_RATIO = 0.20

train_val_pool = [r for r in matched_records if not r["is_test_official"]]
held_out_test = [r for r in matched_records if r["is_test_official"]]

crop_pool = [r for r in train_val_pool if r["is_crop"] == 1]
non_crop_pool = [r for r in train_val_pool if r["is_crop"] == 0]

random.seed(SEED)
random.shuffle(crop_pool)
random.shuffle(non_crop_pool)

n_val_crop = int(len(crop_pool) * VAL_RATIO)
n_val_non_crop = int(len(non_crop_pool) * VAL_RATIO)

val_records = crop_pool[:n_val_crop] + non_crop_pool[:n_val_non_crop]
train_records = crop_pool[n_val_crop:] + non_crop_pool[n_val_non_crop:]
test_records = held_out_test

random.shuffle(train_records)
random.shuffle(val_records)

# Verify zero leakage
train_ids = set(r["filename"] for r in train_records)
val_ids = set(r["filename"] for r in val_records)
test_ids = set(r["filename"] for r in test_records)

assert len(train_ids.intersection(val_ids)) == 0, "Data leakage between train and val!"
assert len(train_ids.intersection(test_ids)) == 0, "Data leakage between train and test!"
assert len(val_ids.intersection(test_ids)) == 0, "Data leakage between val and test!"

print("=" * 70)
print("  REPRODUCIBLE BINARY DATASET SPLIT SUMMARY (ZERO LEAKAGE)")
print("=" * 70)
print(f"  Training Split   : {len(train_records):,} samples "
      f"(Crop: {sum(1 for r in train_records if r['is_crop']==1):,}, "
      f"Non-Crop: {sum(1 for r in train_records if r['is_crop']==0):,})")
print(f"  Validation Split : {len(val_records):,} samples "
      f"(Crop: {sum(1 for r in val_records if r['is_crop']==1):,}, "
      f"Non-Crop: {sum(1 for r in val_records if r['is_crop']==0):,})")
print(f"  Official Test    : {len(test_records):,} samples (Strictly held-out)")
print(f"  Total Partitioned: {len(train_records) + len(val_records) + len(test_records):,} samples")
print("=" * 70)


In [ ]:
# Cell 6: PyTorch Dataset & DataLoaders Definition
# Applies 18-channel normalization (x_norm = (x - mean) / std) on (12, 18) arrays.

class CropHarvestBinaryTorchDataset(Dataset):
    def __init__(
        self,
        records: List[Dict[str, Any]],
        mean: np.ndarray,
        std: np.ndarray,
        preload_into_memory: bool = False
    ):
        self.records = records
        self.mean = mean.astype(np.float32)
        self.std = np.where(std == 0, 1.0, std).astype(np.float32)
        self.preload = preload_into_memory
        self.cached = None

        if self.preload:
            print(f"[DataLoader] Preloading {len(self.records):,} arrays into RAM for high-speed training...")
            t0 = time.time()
            self.cached = []
            for r in self.records:
                with h5py.File(r["filepath"], "r") as h:
                    arr = h["array"][:].astype(np.float32)
                    arr_norm = (arr - self.mean) / self.std
                    arr_norm = np.nan_to_num(arr_norm, nan=0.0, posinf=0.0, neginf=0.0)
                    self.cached.append(arr_norm)
            print(f"[DataLoader] Completed RAM caching in {time.time() - t0:.1f}s.")

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        target = self.records[idx]["is_crop"]
        if self.preload and self.cached is not None:
            data = self.cached[idx]
        else:
            with h5py.File(self.records[idx]["filepath"], "r") as h:
                arr = h["array"][:].astype(np.float32)
            data = (arr - self.mean) / self.std
            data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)

        return torch.from_numpy(data), target

BATCH_SIZE = 64

train_dataset = CropHarvestBinaryTorchDataset(train_records, mean, std, preload_into_memory=False)
val_dataset = CropHarvestBinaryTorchDataset(val_records, mean, std, preload_into_memory=False)
test_dataset = CropHarvestBinaryTorchDataset(test_records, mean, std, preload_into_memory=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2 if IS_KAGGLE else 0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2 if IS_KAGGLE else 0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2 if IS_KAGGLE else 0)

print(f"DataLoaders Ready: {len(train_loader)} train batches, {len(val_loader)} val batches, {len(test_loader)} test batches.")


In [ ]:
# Cell 7: Bi-LSTM Model Architecture & Pre-Flight Sanity Check
# 2-layer Bidirectional LSTM for 12-month multi-spectral temporal phenology modeling.

class CropHarvestBiLSTM(nn.Module):
    def __init__(
        self,
        input_dim: int = 18,
        hidden_dim: int = 64,
        num_layers: int = 2,
        num_classes: int = 2,
        dropout: float = 0.2
    ):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch_size, 12, 18)
        lstm_out, _ = self.lstm(x)  # (batch_size, 12, hidden_dim * 2)
        pooled = lstm_out.mean(dim=1)  # Temporal mean pooling
        normed = self.layer_norm(pooled)
        logits = self.classifier(normed)
        return logits

# Pre-flight Sanity Check
print("[Sanity Check] Verifying model instantiation and synthetic forward/backward pass...")
sanity_model = CropHarvestBiLSTM(input_dim=18, hidden_dim=64, num_layers=2, num_classes=2).to(DEVICE)
total_params = sum(p.numel() for p in sanity_model.parameters() if p.requires_grad)

dummy_x = torch.randn((8, 12, 18), dtype=torch.float32, device=DEVICE)
dummy_y = torch.tensor([0, 1, 0, 1, 1, 0, 1, 0], dtype=torch.long, device=DEVICE)

sanity_model.train()
t0 = time.time()
dummy_out = sanity_model(dummy_x)
dummy_loss = nn.CrossEntropyLoss()(dummy_out, dummy_y)
dummy_loss.backward()
sanity_time_ms = (time.time() - t0) * 1000

print(f"  Model Parameters      : {total_params:,}")
print(f"  Synthetic Batch Forward/Backward: {sanity_time_ms:.2f} ms on {DEVICE}")
if torch.cuda.is_available():
    print(f"  Peak VRAM Allocation : {torch.cuda.max_memory_allocated() / (1024**2):.2f} MB")
print("  --> Pre-flight hardware and model sanity test PASSED!")


In [ ]:
# Cell 8: Model Training Execution (Manual Gate)
# Requirement: Do not start training automatically.
# Set EXECUTE_TRAINING = True when ready to execute on Kaggle GPU.

EXECUTE_TRAINING = False  # Set to True manually to initiate model training

EPOCHS = 15
LEARNING_RATE = 1e-3

if not EXECUTE_TRAINING:
    print("=" * 75)
    print("  [TRAINING GATE: HELD]")
    print("  EXECUTE_TRAINING is set to False.")
    print("  All data integrity and hardware checks have passed.")
    print("  To train the CropHarvest Binary Bi-LSTM model on Kaggle GPU:")
    print("    1. Change `EXECUTE_TRAINING = True` in this cell.")
    print("    2. Re-run this cell.")
    print("=" * 75)
else:
    print(f"Starting Binary Bi-LSTM Training on {DEVICE} for {EPOCHS} epochs...")

    # Class weights for ~69/31 class ratio
    n_train_crop = sum(1 for r in train_records if r["is_crop"] == 1)
    n_train_non_crop = len(train_records) - n_train_crop
    total_train = len(train_records)
    weight_non_crop = total_train / (2.0 * max(1, n_train_non_crop))
    weight_crop = total_train / (2.0 * max(1, n_train_crop))
    class_weights = torch.tensor([weight_non_crop, weight_crop], dtype=torch.float32).to(DEVICE)
    print(f"Loss Class Weights: Non-Crop={weight_non_crop:.3f}, Crop={weight_crop:.3f}")

    model = CropHarvestBiLSTM(input_dim=18, hidden_dim=64, num_layers=2, num_classes=2).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

    best_val_f1 = -1.0
    best_checkpoint_path = OUTPUT_DIR / "best_model.pth"
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_macro_f1": []}

    t_train_start = time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item() * len(batch_y)
            preds = logits.argmax(dim=1)
            correct_train += (preds == batch_y).sum().item()
            total_train += len(batch_y)

        scheduler.step()
        epoch_train_loss = running_loss / total_train
        epoch_train_acc = correct_train / total_train

        # Validation Step
        model.eval()
        val_loss = 0.0
        val_preds, val_targets = [], []
        with torch.no_grad():
            for vx, vy in val_loader:
                vx, vy = vx.to(DEVICE), vy.to(DEVICE)
                v_logits = model(vx)
                v_loss = criterion(v_logits, vy)
                val_loss += v_loss.item() * len(vy)
                val_preds.extend(v_logits.argmax(dim=1).cpu().numpy())
                val_targets.extend(vy.cpu().numpy())

        epoch_val_loss = val_loss / len(val_targets)
        epoch_val_acc = accuracy_score(val_targets, val_preds)
        _, _, epoch_val_f1, _ = precision_recall_fscore_support(val_targets, val_preds, average="macro", zero_division=0)

        history["train_loss"].append(epoch_train_loss)
        history["train_acc"].append(epoch_train_acc)
        history["val_loss"].append(epoch_val_loss)
        history["val_acc"].append(epoch_val_acc)
        history["val_macro_f1"].append(epoch_val_f1)

        is_best = epoch_val_f1 > best_val_f1
        if is_best:
            best_val_f1 = epoch_val_f1
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "val_accuracy": epoch_val_acc,
                "val_macro_f1": epoch_val_f1,
                "config": {"input_dim": 18, "hidden_dim": 64, "num_layers": 2, "num_classes": 2}
            }, best_checkpoint_path)

        star = " ★ BEST" if is_best else ""
        print(
            f"Epoch [{epoch:02d}/{EPOCHS:02d}] "
            f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% | "
            f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}% | "
            f"Val Macro-F1: {epoch_val_f1*100:.2f}%{star}"
        )

    print(f"\nTraining completed in {(time.time() - t_train_start)/60:.2f} minutes.")
    print(f"Best Checkpoint Saved: {best_checkpoint_path.resolve()} (Val Macro-F1: {best_val_f1*100:.2f}%)")


In [ ]:
# Cell 9: Detailed Validation Split Evaluation
# Computes classification report, per-class precision/recall/F1, and confusion matrix.

best_ckpt_file = OUTPUT_DIR / "best_model.pth"

if not best_ckpt_file.exists():
    print("[Notice] Best checkpoint not found. Execute Cell 8 first with EXECUTE_TRAINING = True.")
else:
    eval_model = CropHarvestBiLSTM(input_dim=18, hidden_dim=64, num_layers=2, num_classes=2).to(DEVICE)
    ckpt = torch.load(best_ckpt_file, map_location=DEVICE)
    eval_model.load_state_dict(ckpt["model_state_dict"])
    eval_model.eval()
    print(f"Loaded Best Model from Epoch {ckpt['epoch']} (Val Macro-F1: {ckpt['val_macro_f1']*100:.2f}%)")

    all_vpreds, all_vtargets = [], []
    with torch.no_grad():
        for vx, vy in val_loader:
            vx = vx.to(DEVICE)
            logits = eval_model(vx)
            all_vpreds.extend(logits.argmax(dim=1).cpu().numpy())
            all_vtargets.extend(vy.numpy())

    all_vpreds = np.array(all_vpreds)
    all_vtargets = np.array(all_vtargets)

    val_acc = accuracy_score(all_vtargets, all_vpreds)
    val_prec, val_rec, val_f1, _ = precision_recall_fscore_support(all_vtargets, all_vpreds, average="macro", zero_division=0)
    per_p, per_r, per_f, per_s = precision_recall_fscore_support(all_vtargets, all_vpreds, average=None, zero_division=0)

    print("=" * 70)
    print("  INTERNAL VALIDATION EVALUATION SCORECARD")
    print("=" * 70)
    print(f"  Accuracy  : {val_acc * 100:.2f}%")
    print(f"  Macro-F1  : {val_f1 * 100:.2f}%")
    print(f"  Precision : {val_prec * 100:.2f}%")
    print(f"  Recall    : {val_rec * 100:.2f}%")
    print("-" * 70)
    print(f"  Non-Crop (0): Precision={per_p[0]*100:.2f}%, Recall={per_r[0]*100:.2f}%, F1={per_f[0]*100:.2f}% (Support={per_s[0]:,})")
    print(f"  Crop (1)    : Precision={per_p[1]*100:.2f}%, Recall={per_r[1]*100:.2f}%, F1={per_f[1]*100:.2f}% (Support={per_s[1]:,})")
    print("=" * 70)


In [ ]:
# Cell 10: Untouched Official Held-Out Test Split & Regional Testbeds Evaluation
# Evaluates model generalization on:
# 1. Official Global Test Split (is_test=True, 4,810 samples).
# 2. Togo Regional Testbed (togo-eval.h5, 306 samples, out-of-domain transfer).
# 3. China Regional Testbed (china-crop.h5, 4,397 samples, regional transfer).

def evaluate_testbed_dataset(model: nn.Module, h5_file_path: Path) -> Optional[Dict[str, Any]]:
    if not h5_file_path or not h5_file_path.exists():
        return None
    with h5py.File(h5_file_path, "r") as h:
        x_raw = h["x"][:].astype(np.float32)
        y_raw = h["y"][:].astype(np.int64)

    valid_mask = (y_raw >= 0)
    if not valid_mask.any():
        return None
    x_valid = x_raw[valid_mask]
    y_valid = y_raw[valid_mask]

    std_safe = np.where(std == 0, 1.0, std).astype(np.float32)
    x_norm = (x_valid - mean.astype(np.float32)) / std_safe
    x_norm = np.nan_to_num(x_norm, nan=0.0, posinf=0.0, neginf=0.0)

    ds = torch.utils.data.TensorDataset(torch.from_numpy(x_norm), torch.from_numpy(y_valid))
    loader = DataLoader(ds, batch_size=64, shuffle=False)

    preds, targets = [], []
    with torch.no_grad():
        for bx, by in loader:
            bx = bx.to(DEVICE)
            logits = model(bx)
            preds.extend(logits.argmax(dim=1).cpu().numpy())
            targets.extend(by.numpy())

    preds, targets = np.array(preds), np.array(targets)
    acc = float(accuracy_score(targets, preds))
    prec, rec, f1, _ = precision_recall_fscore_support(targets, preds, average="macro", zero_division=0)
    cm = confusion_matrix(targets, preds).tolist()

    return {
        "accuracy": acc,
        "macro_precision": float(prec),
        "macro_recall": float(rec),
        "macro_f1": float(f1),
        "total_samples": len(targets),
        "confusion_matrix": cm
    }

if not best_ckpt_file.exists():
    print("[Notice] Train the model first in Cell 8 to evaluate test sets.")
else:
    # 1. Global Held-Out Test Evaluation
    all_tpreds, all_ttargets = [], []
    with torch.no_grad():
        for tx, ty in test_loader:
            tx = tx.to(DEVICE)
            logits = eval_model(tx)
            all_tpreds.extend(logits.argmax(dim=1).cpu().numpy())
            all_ttargets.extend(ty.numpy())
    all_tpreds, all_ttargets = np.array(all_tpreds), np.array(all_ttargets)
    global_test_acc = accuracy_score(all_ttargets, all_tpreds)
    _, _, global_test_f1, _ = precision_recall_fscore_support(all_ttargets, all_tpreds, average="macro", zero_division=0)

    # 2. Togo Regional Testbed Evaluation
    togo_eval = evaluate_testbed_dataset(eval_model, ch_paths.get("togo-eval.h5"))

    # 3. China Regional Testbed Evaluation
    china_eval = evaluate_testbed_dataset(eval_model, ch_paths.get("china-crop.h5"))

    print("=" * 75)
    print("  FINAL BENCHMARK EVALUATION SCORECARD")
    print("=" * 75)
    print(f"{'Split / Regional Testbed':<34} | {'Samples':<10} | {'Accuracy':<12} | {'Macro-F1':<12}")
    print("-" * 75)
    print(f"{'Official Global Test (is_test=True)':<34} | {len(all_ttargets):<10} | {global_test_acc*100:>10.2f}% | {global_test_f1*100:>10.2f}%")
    if togo_eval:
        print(f"{'Togo Regional Out-of-Domain Test':<34} | {togo_eval['total_samples']:<10} | {togo_eval['accuracy']*100:>10.2f}% | {togo_eval['macro_f1']*100:>10.2f}%")
    if china_eval:
        print(f"{'China Regional Transfer Test':<34} | {china_eval['total_samples']:<10} | {china_eval['accuracy']*100:>10.2f}% | {china_eval['macro_f1']*100:>10.2f}%")
    print("=" * 75)


In [ ]:
# Cell 11: Confusion Matrix & Temporal Phenology Visualization
# Plots validation confusion matrix and seasonal multi-spectral curves.

if not best_ckpt_file.exists():
    print("[Notice] Visualizations available after model execution.")
else:
    fig, (ax_cm, ax_pheno) = plt.subplots(1, 2, figsize=(14, 5))

    # Confusion Matrix
    cm_val = confusion_matrix(all_vtargets, all_vpreds)
    sns.heatmap(cm_val, annot=True, fmt="d", cmap="Blues", ax=ax_cm,
                xticklabels=["Non-Crop", "Crop"], yticklabels=["Non-Crop", "Crop"])
    ax_cm.set_title(f"Validation Confusion Matrix\n(Acc: {val_acc*100:.2f}%, Macro-F1: {val_f1*100:.2f}%)")
    ax_cm.set_xlabel("Predicted Label")
    ax_cm.set_ylabel("True Ground Truth")

    # Sample Temporal Phenology (Sentinel-2 NIR & Red bands across 12 months)
    sample_crop_idx = [i for i, r in enumerate(val_records) if r["is_crop"] == 1][0]
    sample_non_crop_idx = [i for i, r in enumerate(val_records) if r["is_crop"] == 0][0]

    with h5py.File(val_records[sample_crop_idx]["filepath"], "r") as h:
        arr_crop = h["array"][:]
    with h5py.File(val_records[sample_non_crop_idx]["filepath"], "r") as h:
        arr_non_crop = h["array"][:]

    # S2_B4 (Red) is channel 4, S2_B8 (NIR) is channel 8
    ndvi_crop = (arr_crop[:, 8] - arr_crop[:, 4]) / (arr_crop[:, 8] + arr_crop[:, 4] + 1e-6)
    ndvi_non_crop = (arr_non_crop[:, 8] - arr_non_crop[:, 4]) / (arr_non_crop[:, 8] + arr_non_crop[:, 4] + 1e-6)

    months = list(range(1, 13))
    ax_pheno.plot(months, ndvi_crop, "g-o", label="Crop Phenology (NDVI)")
    ax_pheno.plot(months, ndvi_non_crop, "r--s", label="Non-Crop Phenology (NDVI)")
    ax_pheno.set_title("12-Month Satellite Phenology Signature")
    ax_pheno.set_xlabel("Month of Year")
    ax_pheno.set_ylabel("NDVI Vegetation Index")
    ax_pheno.grid(True, linestyle="--", alpha=0.5)
    ax_pheno.legend()

    plt.tight_layout()
    analysis_plot_path = OUTPUT_DIR / "binary_evaluation_analysis.png"
    plt.savefig(analysis_plot_path, dpi=300)
    plt.show()
    plt.close()
    print(f"Evaluation plots saved to: {analysis_plot_path}")


In [ ]:
# Cell 12: Deliverables & Metrics JSON Export
# Packages trained weights, class mapping, and metrics JSON into models/cropharvest/binary/.

metrics_export_path = OUTPUT_DIR / "metrics.json"

if best_ckpt_file.exists():
    final_metrics_summary = {
        "pipeline": "CropHarvest_Binary_Crop_Classification",
        "task": "is_crop (0: Non-Crop, 1: Crop)",
        "architecture": "CropHarvestBiLSTM",
        "parameters": total_params,
        "input_channels": 18,
        "sequence_length": 12,
        "validation_split": {
            "samples": len(all_vtargets),
            "accuracy": float(val_acc),
            "macro_f1": float(val_f1),
            "macro_precision": float(val_prec),
            "macro_recall": float(val_rec)
        },
        "official_global_test": {
            "samples": len(all_ttargets),
            "accuracy": float(global_test_acc),
            "macro_f1": float(global_test_f1)
        },
        "regional_testbeds": {
            "togo": togo_eval,
            "china": china_eval
        }
    }

    with open(metrics_export_path, "w", encoding="utf-8") as f:
        json.dump(final_metrics_summary, f, indent=2)

    print("=" * 70)
    print("  BINARY PIPELINE DELIVERABLE EXPORT COMPLETE")
    print("=" * 70)
    print(f"  Model Checkpoint : {best_ckpt_file.resolve()}")
    print(f"  Class Mapping    : {(OUTPUT_DIR / 'classes.json').resolve()}")
    print(f"  Metrics JSON     : {metrics_export_path.resolve()}")
    print("=" * 70)
else:
    print(f"Evaluation summary will be exported once training is completed.")
